# Model Training - Terrain Vente Marrakech
Ce notebook entraîne un modèle **XGBoost** pour la prédiction des prix des terrains.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.compose import TransformedTargetRegressor

# Ajouter le chemin vers la racine pour importer les helpers
sys.path.append('../')
from model_training_helpers import split_data, model_pipeline, metric_model, get_features, print_metrics, save_model, preprocess_data

print('Modules chargés avec succès.')

## 1. Chargement des données

In [ ]:
csv_path = '../data/cleaned_data/vente/terrain_vente_cleaned.csv'

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f'Données chargées : {df.shape}')
else:
    print(f'ERREUR : Fichier {csv_path} non trouvé.')

## 2. Préparation

In [ ]:
# Appliquer le prétraitement spécifique aux terrains
df = preprocess_data(df, property_type='terrain')
print(f'Shape après filtrage des outliers : {df.shape}')

# Split des données
X_train, X_test, y_train, y_test = split_data(df, target='prix_num')

# Identification des colonnes
num_features, cat_features = get_features(X_train)
print(f'Features numériques : {len(num_features)}')
print(f'Features catégorielles : {len(cat_features)}')

## 3. Entraînement XGBoost

In [ ]:
print('Entraînement du modèle Terrain (Stacking Haute Performance)...')

# 1. Modèles de base
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV

xgb_reg = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.02, max_depth=8, subsample=0.8, colsample_bytree=0.8, tree_method='hist', n_jobs=-1, random_state=42)
rf_reg = RandomForestRegressor(n_estimators=300, max_depth=12, n_jobs=-1, random_state=42)
hist_reg = HistGradientBoostingRegressor(max_iter=500, max_depth=10, random_state=42)

# 2. Stacking
estimators = [
    ('xgb', model_pipeline(num_features, cat_features, xgb_reg)),
    ('rf', model_pipeline(num_features, cat_features, rf_reg)),
    ('hist', model_pipeline(num_features, cat_features, hist_reg))
]

stacking = StackingRegressor(
    estimators=estimators,
    final_estimator=RidgeCV(),
    cv=2,
    n_jobs=-1
)

# 3. Encapsulation Log-Target
pipeline = TransformedTargetRegressor(
    regressor=stacking,
    func=np.log1p,
    inverse_func=np.expm1
)

print("Lancement de l'entraînement du Stacking Terrains...")
pipeline.fit(X_train, y_train)
print('Modèle Terrain Haute Performance prêt.')

## 4. Évaluation

In [ ]:
y_pred = pipeline.predict(X_test)
metrics = metric_model(y_test, y_pred, model_name='XGBoost Terrains')
print_metrics(metrics, model_name='XGBoost Terrains')

## 5. Sauvegarde

In [ ]:
save_model(pipeline, 'terrain_vente_final_model.joblib')
print('Modèle sauvegardé.')